# What is Randomized Search CV?

Randomized Search CV is an efficient method to find good hyperparameters for a model.

- Instead of trying every possible combination (like Grid Search), it samples a fixed number of random combinations from the grid.

- Uses cross-validation to measure performance reliably.

- Much faster than Grid Search, especially with large grids.



## The Math Behind It

Suppose we have:

- Model $f(x;\theta)$ with parameters $\theta$.

- Hyperparameters $h = (h_1, h_2, ..., h_k)$.

- Grid $G$ of possible values.



Randomized Search CV steps:

1. Randomly sample $N$ combinations $h$ from $G$.

2. For each sampled $h$:

   - Train the model with $h$.

   - Compute cross-validation score:

     $$CV(h) = \frac{1}{K} \sum_{i=1}^K Accuracy_i(h)$$

3. Pick the best performing combination:

   $$h^* = \arg\max_{h} CV(h)$$



## Example

Suppose we’re tuning an SVM:

- Hyperparameters:

  - Kernel = {linear, rbf}

  - C = {0.1, 1, 10}

  - Gamma = {0.01, 0.1}

- Grid = all possible combos = $2 \times 3 \times 2 = 12$

- Randomized Search CV might sample 5 random combinations out of 12.



Randomized Search CV will:

- Train SVM for each of the 5 sampled settings.

- Use K-fold CV (say K=5) → 5 evaluations per setting.

- Pick best performing combination.





## Visual (Conceptual)

Imagine a grid of possible hyperparameter combinations.
- Randomized Search picks a few random points to try.
- Faster, but may miss the absolute best if not enough samples.



## Comparison

- **Grid Search CV:** Tries all combos blindly → exhaustive but slow.

- **Randomized Search CV:** Tries a random subset → much faster, good for large grids.

- **Bayesian Optimization:** Explores smartly, learns from previous results, fastest for large spaces.


In [1]:
import pandas as pd

In [3]:
df=pd.read_csv("balanced_fraud_detection_data.csv")

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [6]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier 
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [7]:
models = {
    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7],
            'weights': ['uniform', 'distance'],
            'algorithm': ['auto', 'ball_tree'],
            'leaf_size': [30, 50]
        }
    },
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5],
            'criterion': ['gini', 'entropy']
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(solver='liblinear', random_state=42),
        'params': {
            'C': [0.1, 1, 10],
            'penalty': ['l1', 'l2'],
            'fit_intercept': [True, False],
            'max_iter': [100, 200]
        }
    }
}


In [9]:
results = {}
for name, mp in models.items():
    print(f"\nRunning RandomizedSearchCV for {name}...")
    grid = RandomizedSearchCV(mp['model'], mp['params'], cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    results[name] = {
        'best_score': grid.best_score_,
        'best_params': grid.best_params_,
        'test_score': grid.score(X_test, y_test)
    }
    print(f"Best CV Score: {grid.best_score_:.4f}")
    print(f"Best Params: {grid.best_params_}")
    print(f"Test Score: {grid.score(X_test, y_test):.4f}")

print("\nSummary of Results:")
for name, res in results.items():
    print(f"{name}: Best CV Score={res['best_score']:.4f}, Test Score={res['test_score']:.4f}, Best Params={res['best_params']}")


Running RandomizedSearchCV for KNN...
Best CV Score: 0.9280
Best Params: {'weights': 'distance', 'n_neighbors': 7, 'leaf_size': 30, 'algorithm': 'auto'}
Test Score: 0.9210

Running RandomizedSearchCV for RandomForest...
Best CV Score: 0.9611
Best Params: {'n_estimators': 100, 'min_samples_split': 5, 'max_depth': 20, 'criterion': 'entropy'}
Test Score: 0.9565

Running RandomizedSearchCV for LogisticRegression...
Best CV Score: 0.8826
Best Params: {'penalty': 'l1', 'max_iter': 100, 'fit_intercept': True, 'C': 0.1}
Test Score: 0.8730

Summary of Results:
KNN: Best CV Score=0.9280, Test Score=0.9210, Best Params={'weights': 'distance', 'n_neighbors': 7, 'leaf_size': 30, 'algorithm': 'auto'}
RandomForest: Best CV Score=0.9611, Test Score=0.9565, Best Params={'n_estimators': 100, 'min_samples_split': 5, 'max_depth': 20, 'criterion': 'entropy'}
LogisticRegression: Best CV Score=0.8826, Test Score=0.8730, Best Params={'penalty': 'l1', 'max_iter': 100, 'fit_intercept': True, 'C': 0.1}


In [10]:
# it took 30 sec for randomized search , it takes very less time compared to grid search
# it randomly provides a subset of hyperparameters to try, making it more efficient
# we can choose the best combination of hyperparameters from rerunning the program multiple times